# 🌐 Macro — Data Collector
**Data source:** Yahoo Finance &nbsp;·&nbsp; `yfinance` Python library

This notebook lets you download macro-level market data across five categories.

### Available categories
| Category | What it contains |
|---|---|
| 📉 Rates & Yields | US Treasury yields (2Y, 5Y, 10Y, 30Y) and T-bill rate |
| ⚡ Volatility | VIX (equity fear gauge), NASDAQ VIX, Gold VIX, Oil VIX |
| 🛢️ Commodities | Gold, silver, crude oil (WTI & Brent), natural gas, agricultural |
| 💱 Foreign Exchange | Major currency pairs vs USD |
| ₿ Crypto | Bitcoin, Ethereum, BNB, Solana |

### What you can download
- 📊 **Prices** — close price / level history
- 📈 **Returns** — period-over-period percentage changes

### How to use this notebook
1. **Run the Setup cell** first
2. Select the categories and series you want
3. Set the date range and frequency
4. Click **Download Data**


In [ ]:
# ── Setup — Run this cell first ─────────────────────────────────────────────
import sys
import datetime
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from src.collect import download_prices
from src.transform import resample_prices, calculate_returns
from src.export import save_to_excel

print("✅ Setup complete — continue to the next cell.")


---
## Step 1 — Select Macro Series

Tick the categories you want, then select the individual series within each.


In [ ]:
# ── Series definitions ───────────────────────────────────────────────────────

MACRO_SERIES = {
    "📉 Rates & Yields": {
        "^IRX":  "US 13-Week T-Bill Yield",
        "^FVX":  "US 5-Year Treasury Yield",
        "^TNX":  "US 10-Year Treasury Yield",
        "^TYX":  "US 30-Year Treasury Yield",
    },
    "⚡ Volatility": {
        "^VIX":  "CBOE Volatility Index (S&P 500)",
        "^VXN":  "CBOE NASDAQ Volatility Index",
        "^GVZ":  "CBOE Gold Volatility Index",
        "^OVX":  "CBOE Crude Oil Volatility Index",
    },
    "🛢️ Commodities": {
        "GC=F":  "Gold (USD/oz)",
        "SI=F":  "Silver (USD/oz)",
        "CL=F":  "Crude Oil — WTI (USD/barrel)",
        "BZ=F":  "Crude Oil — Brent (USD/barrel)",
        "NG=F":  "Natural Gas (USD/MMBtu)",
        "HG=F":  "Copper (USD/lb)",
        "ZW=F":  "Wheat (USD/bushel)",
        "ZC=F":  "Corn (USD/bushel)",
        "ZS=F":  "Soybeans (USD/bushel)",
    },
    "💱 Foreign Exchange": {
        "EURUSD=X": "EUR / USD",
        "GBPUSD=X": "GBP / USD",
        "JPYUSD=X": "JPY / USD",
        "CNYUSD=X": "CNY / USD",
        "CHFUSD=X": "CHF / USD",
        "CADUSD=X": "CAD / USD",
        "AUDUSD=X": "AUD / USD",
        "DX-Y.NYB": "US Dollar Index (DXY)",
    },
    "₿ Crypto": {
        "BTC-USD": "Bitcoin (USD)",
        "ETH-USD": "Ethereum (USD)",
        "BNB-USD": "BNB (USD)",
        "SOL-USD": "Solana (USD)",
    },
}

# ── Build widgets ─────────────────────────────────────────────────────────────

category_widgets = {}

for cat_name, series_dict in MACRO_SERIES.items():
    options = [f"{ticker}  —  {desc}" for ticker, desc in series_dict.items()]
    sel = widgets.SelectMultiple(
        options=options,
        value=[],
        layout=widgets.Layout(width="520px", height=f"{min(len(options) * 26 + 10, 170)}px"),
    )
    toggle = widgets.Checkbox(value=False, description=cat_name,
                               style={"description_width": "initial"})
    category_widgets[cat_name] = {"toggle": toggle, "select": sel, "series": series_dict}

# Show all
for cat_name, w in category_widgets.items():
    display(
        w["toggle"],
        widgets.HTML("<div style='margin-left:24px'>"),
        w["select"],
        widgets.HTML("</div><br>"),
    )

display(widgets.HTML(
    "<small style='color:#666'>Hold <b>Ctrl</b> (Windows) or <b>Cmd</b> (Mac) to select multiple items.</small>"
))


---
## Step 2 — Date Range & Frequency


In [ ]:
start_date = widgets.DatePicker(
    description="Start date:",
    value=datetime.date(2010, 1, 1),
    style={"description_width": "90px"},
)
end_date = widgets.DatePicker(
    description="End date:",
    value=datetime.date.today(),
    style={"description_width": "90px"},
)
frequency = widgets.Dropdown(
    options=["Daily", "Monthly", "Quarterly", "Yearly"],
    value="Monthly",
    description="Frequency:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="220px"),
)

display(
    widgets.HTML("<b>Date range:</b>"),
    widgets.HBox([start_date, end_date]),
    widgets.HTML("<br><b>Frequency:</b>"),
    frequency,
)


---
## Step 3 — Choose Data to Download


In [ ]:
cb_prices  = widgets.Checkbox(value=True,  description="📊  Prices / Levels")
cb_returns = widgets.Checkbox(value=True,  description="📈  Returns (percentage changes)")

display(cb_prices, cb_returns)


---
## Step 4 — Download & Save


In [ ]:
output_file = widgets.Text(
    value="market_data.xlsx",
    description="File name:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="320px"),
)

download_btn = widgets.Button(
    description="⬇  Download Data",
    button_style="success",
    layout=widgets.Layout(width="200px", height="40px"),
)

out = widgets.Output()
display(output_file, download_btn, out)


def on_download(b):
    with out:
        clear_output(wait=True)

        # ── Collect selected tickers ──────────────────────────────────────────
        selected_tickers = []
        for cat_name, w in category_widgets.items():
            if w["toggle"].value:
                for item in w["select"].value:
                    ticker = item.split("  —  ")[0].strip()
                    selected_tickers.append(ticker)
            elif w["select"].value:
                # Allow selection without toggle (just in case)
                for item in w["select"].value:
                    ticker = item.split("  —  ")[0].strip()
                    selected_tickers.append(ticker)

        # Remove duplicates while preserving order
        seen = set()
        tickers = [t for t in selected_tickers if not (t in seen or seen.add(t))]

        if not tickers:
            print("❌  Please select at least one series in Step 1.")
            return

        if not cb_prices.value and not cb_returns.value:
            print("❌  Please select at least one data type in Step 3.")
            return

        start = str(start_date.value)
        end   = str(end_date.value)
        freq  = frequency.value

        fname = output_file.value.strip() or "market_data.xlsx"
        if not fname.endswith(".xlsx"):
            fname += ".xlsx"

        output_path = Path("..") / "data" / fname
        sheets = {}

        print(f"Series   : {', '.join(tickers)}")
        print(f"Period   : {start}  →  {end}")
        print(f"Frequency: {freq}")
        print()

        print("📊  Downloading macro data…")
        try:
            prices = download_prices(tickers, start, end, freq)
            prices = resample_prices(prices, freq)
            print(f"    {len(prices)} rows × {len(prices.columns)} series")

            if cb_prices.value:
                sheets["Macro Prices"] = prices
            if cb_returns.value:
                sheets["Macro Returns"] = calculate_returns(prices)

            print("    ✅  Done")
        except Exception as e:
            print(f"    ❌  Error: {e}")

        if not sheets:
            print("\n❌  No data collected. Nothing saved.")
            return

        print(f"\n💾  Saving to Excel…")
        try:
            save_to_excel(sheets, output_path)
            print(f"    ✅  Saved: {output_path.resolve()}")
            print(f"    Sheets: {', '.join(sheets.keys())}")
        except Exception as e:
            print(f"    ❌  Could not save: {e}")
            return

        from datetime import datetime as dt
        today = dt.today().strftime("%d %B %Y")
        print()
        print("─" * 60)
        print("📋  DATA SOURCE — copy this into your assignment")
        print("─" * 60)
        print(f"Source     : Yahoo Finance (finance.yahoo.com)")
        print(f"Series     : {', '.join(tickers)}")
        print(f"Period     : {start} to {end}  |  Frequency: {freq}")
        print(f"Downloaded : {today}")
        print(f"Tool       : yfinance Python library (pypi.org/project/yfinance)")
        print("─" * 60)


download_btn.on_click(on_download)
